In [ ]:

!pip install -q openai faiss-cpu pypdf tiktoken numpy
from openai import OpenAI
import os
import getpass
imporimportt numpy as np
import faiss
import tiktoken

from google.colab import files
from pypdf import PdfReader
api_key = getpass.getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=api_key)
# 2. FILE UPLOAD
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"Uploaded file: {file_name}")
# 3. TEXT EXTRACTION
def extract_text(file_path: str) -> str:
    file_path_lower = file_path.lower()

    if file_path_lower.endswith(".pdf"):
        reader = PdfReader(file_path)
        pages = []

        for page in reader.pages:
            page_text = page.extract_text() or ""
            pages.append(page_text)

        return "\n".join(pages).strip()

    if file_path_lower.endswith(".txt") or file_path_lower.endswith(".md"):
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read().strip()

    raise ValueError("Unsupported file format. Please upload PDF, TXT, or MD.")
# 4. CHUNKING
def chunk_text(
    text: str,
    max_tokens: int = 500,
    overlap: int = 100,
    encoding_name: str = "cl100k_base"
) -> list[str]:
    if not text.strip():
        return []
    encoding = tiktoken.get_encoding(encoding_name)
    tokens = encoding.encode(text)
    chunks = []
    start = 0
    step = max_tokens - overlap
    if step <= 0:
        raise ValueError("overlap must be smaller than max_tokens")
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunks.append(encoding.decode(chunk_tokens))
        start += step

    return chunks
# 5. EMBEDDINGS

def get_embeddings(
    text_list: list[str],
    model: str = "text-embedding-3-small",
    batch_size: int = 100
) -> list[list[float]]:
    embeddings = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]

        response = client.embeddings.create(
            model=model,
            input=batch
        )

        embeddings.extend(item.embedding for item in response.data)

    return embeddings
# 6. BUILD KNOWLEDGE INDEX
document_text = extract_text(file_name)
chunks = chunk_text(document_text, max_tokens=500, overlap=100)

if not chunks:
    raise ValueError("No text could be extracted from the uploaded file.")

chunk_embeddings = get_embeddings(chunks)
embedding_matrix = np.array(chunk_embeddings, dtype="float32")

dimension = embedding_matrix.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print(f"Document indexed successfully. Total chunks stored: {len(chunks)}")
# 7. RETRIEVAL
def retrieve_relevant_chunks(query: str, k: int = 3) -> list[str]:
    query_embedding = get_embeddings([query])[0]
    query_vector = np.array([query_embedding], dtype="float32")

    distances, indices = index.search(query_vector, k)

    results = []
    for idx in indices[0]:
        if 0 <= idx < len(chunks):
            results.append(chunks[idx])

    return results
# 8. QUESTION ANSWERING
def answer_question(query: str, k: int = 3, model: str = "gpt-4.1-mini") -> str:
    relevant_chunks = retrieve_relevant_chunks(query, k=k)
    context = "\n\n".join(relevant_chunks)

    prompt = f"""
You are a helpful knowledge assistant.

Answer the user's question only using the context below.
If the answer is not in the context, say:
"I could not find the answer in the uploaded document."

Write the answer in a clear and natural way.

Context:
{context}

Question:
{query}
""".strip()

    response = client.responses.create(
        model=model,
        input=prompt
    )

    return response.output_text
# 9. CHAT LOOP
while True:
    user_question = input("\nAsk a question (or type 'exit'): ").strip()

    if user_question.lower() == "exit":
        print("Session ended.")
        break

    if not user_question:
        print("Please enter a question.")
        continue

    answer = answer_question(user_question, k=3)
    print("\nAnswer:\n")
    print(answer)

 ------------------------CHATBOT WITH GRADIO UI----------------------

In [ ]:
import os
import numpy as np
import faiss
import tiktoken
import gradio as gr
import getpass
from pypdf import PdfReader
from openai import OpenAI
# 1. SETUP
api_key = getpass.getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=api_key)
# 2. GLOBAL STORAGE
chunks = []
index = None
current_file_name = None
# 3. TEXT EXTRACTION

def extract_text(file_path: str) -> str:
    file_path_lower = file_path.lower()

    if file_path_lower.endswith(".pdf"):
        reader = PdfReader(file_path)
        pages = []

        for page in reader.pages:
            page_text = page.extract_text() or ""
            pages.append(page_text)

        return "\n".join(pages).strip()

    if file_path_lower.endswith(".txt") or file_path_lower.endswith(".md"):
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read().strip()

    raise ValueError("Unsupported file format. Please upload PDF, TXT, or MD.")
# 4. CHUNKING
def chunk_text(
    text: str,
    max_tokens: int = 500,
    overlap: int = 100,
    encoding_name: str = "cl100k_base"
) -> list[str]:
    if not text.strip():
        return []
    encoding = tiktoken.get_encoding(encoding_name)
    tokens = encoding.encode(text)
    chunks_local = []
    start = 0
    step = max_tokens - overlap

    if step <= 0:
        raise ValueError("overlap must be smaller than max_tokens")

    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunks_local.append(encoding.decode(chunk_tokens))
        start += step

    return chunks_local
# 5. EMBEDDINGS
def get_embeddings(
    text_list: list[str],
    model: str = "text-embedding-3-small",
    batch_size: int = 100
) -> list[list[float]]:
    embeddings = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]

        response = client.embeddings.create(
            model=model,
            input=batch
        )

        embeddings.extend(item.embedding for item in response.data)

    return embeddings
# 6. BUILD KNOWLEDGE INDEX
def build_knowledge_base(file):
    global chunks, index, current_file_name

    if file is None:
        return "Please upload a file first."

    try:
        file_path = file.name
        current_file_name = os.path.basename(file_path)

        document_text = extract_text(file_path)
        chunks = chunk_text(document_text, max_tokens=500, overlap=100)

        if not chunks:
            index = None
            return "No text could be extracted from the uploaded file."

        chunk_embeddings = get_embeddings(chunks)
        embedding_matrix = np.array(chunk_embeddings, dtype="float32")

        dimension = embedding_matrix.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(embedding_matrix)

        return f"Document indexed successfully: {current_file_name}\nTotal chunks stored: {len(chunks)}"

    except Exception as e:
        index = None
        chunks = []
        return f"Error while processing file: {str(e)}"
# 7. RETRIEVAL
def retrieve_relevant_chunks(query: str, k: int = 3) -> list[str]:
    global index, chunks

    if index is None or not chunks:
        return []

    query_embedding = get_embeddings([query])[0]
    query_vector = np.array([query_embedding], dtype="float32")

    distances, indices = index.search(query_vector, min(k, len(chunks)))

    results = []
    for idx in indices[0]:
        if 0 <= idx < len(chunks):
            results.append(chunks[idx])

    return results
# 8. QUESTION ANSWERING
def answer_question(query: str, k: int = 3, model: str = "gpt-4.1-mini") -> str:
    relevant_chunks = retrieve_relevant_chunks(query, k=k)

    if not relevant_chunks:
        return "Please upload and index a document first."

    context = "\n\n".join(relevant_chunks)

    prompt = f"""
You are a helpful knowledge assistant.

Answer the user's question only using the context below.
If the answer is not in the context, say:
"I could not find the answer in the uploaded document."

Write the answer in a clear and natural way.

Context:
{context}

Question:
{query}
""".strip()

    response = client.responses.create(
        model=model,
        input=prompt
    )

    return response.output_text
# 10. GRADIO UI

import gradio as gr

def build_index(uploaded_file):
    global document_text, chunks, chunk_embeddings, embedding_matrix, dimension, index

    if uploaded_file is None:
        return "Please upload a PDF, TXT, or MD file."

    file_path = uploaded_file.name

    try:
        document_text = extract_text(file_path)
        chunks = chunk_text(document_text, max_tokens=500, overlap=100)

        if not chunks:
            return "No text could be extracted from the uploaded file."

        chunk_embeddings = get_embeddings(chunks)
        embedding_matrix = np.array(chunk_embeddings, dtype="float32")

        dimension = embedding_matrix.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(embedding_matrix)

        return f"Document indexed successfully. Total chunks stored: {len(chunks)}"

    except Exception as e:
        return f"Error: {str(e)}"


def chat_with_pdf(message, history):
    global index, chunks

    if "index" not in globals() or index is None:
        history.append((message, "Please upload and process a document first."))
        return history, ""

    try:
        answer = answer_question(message, k=3)
        history.append((message, answer))
        return history, ""
    except Exception as e:
        history.append((message, f"Error: {str(e)}"))
        return history, ""


with gr.Blocks(title="PDF Chatbot") as demo:
    gr.Markdown("## Chat with your PDF")
    gr.Markdown("Upload a PDF/TXT/MD file, process it, and then ask questions.")

    with gr.Row():
        file_input = gr.File(
            label="Upload your document",
            file_types=[".pdf", ".txt", ".md"]
        )
        process_btn = gr.Button("Process Document")

    status_output = gr.Textbox(
        label="Status",
        interactive=False
    )

    chatbot = gr.Chatbot(label="Conversation", height=450)

    with gr.Row():
        msg = gr.Textbox(
            label="Ask a question",
            placeholder="Type your question here..."
        )
        send_btn = gr.Button("Send")

    process_btn.click(
        fn=build_index,
        inputs=file_input,
        outputs=status_output
    )

    send_btn.click(
        fn=chat_with_pdf,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

    msg.submit(
        fn=chat_with_pdf,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

demo.launch(debug=True)